# 12b — The guarded sweep (E15 completion; director package spec step 0; VERSION switch added v0.17; manifest v4 added v0.20)

**VERSION = "v4"** (cell 1) runs the per-block-floor sweeps on the re-registered design (`runs_v4/`, `spec/manifest_v4.csv`: seven scenarios × two refugia realizations = 14 formulations, after 12 has produced their anchors and twins). What changed against v3.1 — every change keyed on a NEW manifest column, so `VERSION = "v3.1"` still reproduces the prior registration from its own manifest (resumable evidence):

- the block structure the floors guard comes from the manifest's `block_structure` (five blocks under v4 — core habitat, structural connectivity, climate corridors, carbon, biodiversity — against the four of `spec/e_round_v13.json` before); every block feature is asserted present in the ingested stack, and `guard_meta.json` records the blocks used;
- the refugia layer per formulation from `macrorefugia_path` (the SSP245 patch is no longer a literal), the ingested ssp585 layer asserted against it;
- the EFG block folder from `efg_block_version` (v3 → `iucn_efg_v3`);
- the MAA spot-check seed from `e12_seed` (the E-round record's seed stands in where the column is absent).

Under v4 the guarded members ARE the ensemble for every formulation but the reference cell (12 runs the unguarded MGA on the reference cell only, at 2 / 5 / 10 %). Run order for the v4 record: **11c → 12 → 12b → 12c → 13 → 15 → 15b → 19 → 20 → 21**, each run by Ethan in VS Code (no headless runs). Naming convention: things by what they are, codes in parentheses.

`VERSION = "v3.1"` runs the sweeps on the curated-EFG-block re-solve (`runs_v3.1/`, 12 design formulations); `"v1"` = the as-frozen record (`runs/`, done 2026-09-04). Measured v1 cost: 52–76 min per formulation under contention.

Spec v0.14.1 authorizes extending the per-block-floor MGA sweeps from {S0, S4} (notebook 16)
to **all frozen formulations**, plus the queued **guarded MAA spot-check on S0**. Guarded
semantics = the applied headline ("no value theme left more than 5% behind"); the aggregate
band remains the Claim-A estimand.

Per formulation, into `analyses/y2y/runs<_version>/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `mga_guard_g05.tif` + `certificates_guard.csv` | 50 guarded MGA members | band g=5% + per-block floors (the manifest's blocks: five under v4) capture_b ≥ 0.95·anchor_b |
| `guard_meta.json` | re-solved anchor vs the frozen record, the blocks floored, timings | — |
| S0 only: `maa_guard_g05.tif` + `certificates_maa_guard.csv` | guarded MAA spot-check | E12 seed (the manifest's `e12_seed` under v4; 20260902 from the E-round record before) + the same floors |

**Anchors, k-best pools and twins STAND** — floors are anchor-relative, so the anchor is
re-solved only to obtain the compiled model + `x` (asserted ≤1e-3 relative from the frozen
`formulation_meta.json` objective). Byte-compatible with 16's S0/S4 outputs (skipped here).
Measured cost 11–15 min per formulation (floors *shrink* the search) ⇒ **~3 h serial for the
12 open formulations** — the spec's "8–10 h" is superseded by this measurement. Fully
resumable; **live internet throughout** (WLS). Kernel `R (y2y)`.


*Renumbered 18 → 12b on 2026-09-17 (Ethan: numeric order = run order; the sweep runs right after 12 because the guarded band is the estimand under manifest v4).*


In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # the ACTIVE version's stack manifest (aligned_stack_v4/manifest.json under v4)

# ---- VERSION switch (study plan v0.17; supersede, never delete) --------------------------------------------
# "v1" = the as-frozen 2026-08-30 record (spec/manifest.csv, runs/); "v3.1" = the curated EFG block with window-derived
# targets (spec/manifest_v3.1.csv, runs_v3.1/; 12 design formulations); "v4" = the re-registered design
# (spec/manifest_v4.csv, runs_v4/; 14 formulations). config.Y2Y_VERSION must agree (the ingested stack is asserted
# against the manifest in cell 2). Every v4 behaviour is keyed on a manifest COLUMN, never on the version string, so an
# earlier manifest reproduces its own record unchanged.
VERSION <- "v4"      # v4 = manifest v4, study plan v0.20 -- five blocks, seven scenarios, floored refugia + squared transboundary current; v3.1 = the prior registration
MANIFEST_REL <- if (VERSION == "v1") "analyses/y2y/spec/manifest.csv" else sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION)
FREEZE_REL   <- if (VERSION == "v1") "analyses/y2y/spec/manifest_freeze.sha256" else sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL     <- if (VERSION == "v1") "analyses/y2y/runs" else sprintf("analyses/y2y/runs_%s", VERSION)
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
for (col in c("kbest_ref", "twin_ref")) {           # v3+ has no pool/twin pointers: an all-empty column reads as NA
  if (!col %in% names(MAN)) MAN[[col]] <- ""
  MAN[[col]] <- ifelse(is.na(MAN[[col]]), "", as.character(MAN[[col]]))
}
stopifnot(nrow(MAN) %in% c(12, 14))
# the EFG block folder: the manifest's efg_block_version when it carries one (v3 -> iucn_efg_v3; shared by v3, v3.1 and v4),
# else the pre-column rule (v1 -> iucn_efg; minor versions share the block)
EFG_SUBDIR_EXPECTED <- if ("efg_block_version" %in% names(MAN)) {
  stopifnot("formulations disagree on efg_block_version -- STOP" = length(unique(MAN$efg_block_version)) == 1)
  paste0("iucn_efg_", MAN$efg_block_version[1])
} else if (VERSION == "v1") "iucn_efg" else paste0("iucn_efg_", sub("\\..*$", "", VERSION))
# verify the freeze hash before solving against it
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot("manifest does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig))
cat(sprintf("VERSION %s: %s verified against freeze hash %s... (%d formulations)\n", VERSION, basename(MANIFEST_REL), substr(dig, 1, 16), nrow(MAN)))
RUNS <- file.path(PROJ, RUNS_REL)

# ---- the refugia layer per formulation -----------------------------------------------------------------------
# manifest v4 registers it per row (column macrorefugia_path: the floored 1/v layer of the v4 stack for ssp585, its 245
# realization for ssp245); earlier manifests carry no column, so the pre-v4 literals stand in.
REFUGIA_585_FALLBACK <- "input_data/aligned_stack/climate_type_macrorefugia.tif"
REFUGIA_245_FALLBACK <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
refugia_path_for <- function(row) {
  p <- if ("macrorefugia_path" %in% names(row)) trimws(as.character(row$macrorefugia_path)) else NA_character_
  if (length(p) == 1 && !is.na(p) && nzchar(p)) return(p)
  if (grepl("^ssp245", row$climate_level)) REFUGIA_245_FALLBACK else REFUGIA_585_FALLBACK
}
i245 <- which(grepl("^ssp245", MAN$climate_level))
paths245 <- unique(vapply(i245, function(i) refugia_path_for(MAN[i, ]), character(1)))
stopifnot("ssp245 formulations disagree on their refugia realization -- STOP" = length(paths245) <= 1)
REFUGIA_245_PATH <- if (length(paths245) == 1) paths245 else REFUGIA_245_FALLBACK   # patched into the 245 base context
cat(sprintf("refugia (ssp245, %d formulations): %s\n", length(i245), REFUGIA_245_PATH))

# ---- the value blocks --------------------------------------------------------------------------------------------
# the manifest's block_structure when it carries one (v4: five blocks -- core habitat, structural connectivity, climate
# corridors, carbon, biodiversity; asserted identical on every row), else the four PROACT blocks of the E-round record
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))   # E12 seeds (+ the pre-v4 blocks)
BLOCKS <- if ("block_structure" %in% names(MAN)) {
  stopifnot("formulations disagree on block_structure -- the manifest is not one design; STOP" = length(unique(MAN$block_structure)) == 1)
  lapply(jsonlite::fromJSON(MAN$block_structure[1]), unlist)
} else lapply(ER$e17_t3$blocks, unlist)
FLOOR_G <- 0.05                                   # capture_b >= (1 - FLOOR_G) * anchor_b
MAA_SPOTCHECK <- "s0_ssp585_theta5"
cat(sprintf("floors on %d blocks: %s | floor g = %g\n", length(BLOCKS), paste(names(BLOCKS), collapse = ", "), FLOOR_G))


manifest refreshed from config.py (analysis=y2y) -> input_data/aligned_stack_v4/manifest.json
VERSION v4: manifest_v4.csv verified against freeze hash c0d4c594af720819... (14 formulations)
refugia (ssp245, 7 formulations): input_data/aligned_stack_v4/climate_realizations/macrorefugia_245_2071_2100.tif
floors on 5 blocks: core_habitat, structural_connectivity, climate_corridors, carbon, biodiversity | floor g = 0.05


In [2]:
# ---- two ingested base contexts, built ONCE (one per climate level) ------------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))

ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REFUGIA_245_PATH   # the realization the manifest registers (cell 1)
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_used <- ctx585$layers$path[ctx585$layers$role == "feature_efg"]
stopifnot("manifest.json enumerates a different EFG block than VERSION expects -- check config.EFG_SUBDIR" =
            length(efg_used) > 0 && all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), efg_used)))
# the 585 context ingests whatever stack config.Y2Y_VERSION points at (aligned_stack_v4/ under v4): its refugia layer must
# be the one the manifest registers for the ssp585 formulations, else VERSION and config disagree
refugia_ingested <- ctx585$layers$path[ctx585$layers$name == "climate_type_macrorefugia"]
paths585 <- unique(vapply(which(!grepl("^ssp245", MAN$climate_level)), function(i) refugia_path_for(MAN[i, ]), character(1)))
same_file <- function(a, b) {
  abs_of <- function(p) normalizePath(if (grepl("^/", p)) p else file.path(PROJ, p), mustWork = FALSE)
  identical(abs_of(a), abs_of(b))
}
stopifnot("the ingested refugia layer is not the one the manifest registers for ssp585 -- config.Y2Y_VERSION and VERSION disagree; STOP" =
            length(paths585) == 1 && length(refugia_ingested) == 1 && same_file(paths585, refugia_ingested))
# every block feature must be a continuous feature the ingested stack carries (replaces the fixed four-name check)
cont_names <- ctx585$layers$name[ctx585$layers$role == "feature_continuous"]
stopifnot("a block names a feature the ingested stack does not carry -- block_structure vs manifest.json; STOP" =
            length(BLOCKS) > 0 && all(unlist(BLOCKS) %in% cont_names))
cat(sprintf("blocks (%d): %s | %d features verified against the ingested stack\n", length(BLOCKS),
            paste(names(BLOCKS), collapse = ", "), length(unlist(BLOCKS))))
cat(sprintf("EFG block: %d features from %s/\n", length(efg_used), EFG_SUBDIR_EXPECTED))
cat(sprintf("refugia: ssp585 %s | ssp245 %s\n", refugia_ingested, REFUGIA_245_PATH))
cat("base contexts ready (585 canonical; 245 with the realization layer patched)\n")

base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector),
                               t = jsonlite::fromJSON(row$target_vector))


prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 feature

In [3]:
# ---- DRY PLAN (no solves): what this run will do -------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  st <- if (file.exists(file.path(cd, "mga_guard_g05.tif"))) "done" else "TODO"
  maa <- if (row$formulation_id == MAA_SPOTCHECK)
           (if (file.exists(file.path(cd, "maa_guard_g05.tif"))) " | maa-guard: done" else " | maa-guard: TODO") else ""
  cat(sprintf("%-22s %-9s mga-guard: %-5s%s\n", row$formulation_id,
              sub("_2071_2100", "", row$climate_level), st, maa))
}


s0_ssp585_theta5       ssp585    mga-guard: TODO  | maa-guard: TODO
s1_ssp585_theta5       ssp585    mga-guard: TODO 
s2_ssp585_theta5       ssp585    mga-guard: TODO 
s2c_ssp585_theta5      ssp585    mga-guard: TODO 
s3_ssp585_theta5       ssp585    mga-guard: TODO 
s4_ssp585_theta3       ssp585    mga-guard: TODO 
s5_ssp585_theta5       ssp585    mga-guard: TODO 
s0_ssp245_theta5       ssp245    mga-guard: TODO 
s1_ssp245_theta5       ssp245    mga-guard: TODO 
s2_ssp245_theta5       ssp245    mga-guard: TODO 
s2c_ssp245_theta5      ssp245    mga-guard: TODO 
s3_ssp245_theta5       ssp245    mga-guard: TODO 
s4_ssp245_theta3       ssp245    mga-guard: TODO 
s5_ssp245_theta5       ssp245    mga-guard: TODO 


In [4]:
# ---- the guarded-sweep runner (one artifact; skipped when its tif exists) -------------------
run_guard <- function(row, kind = c("mga", "maa")) {
  kind <- match.arg(kind)
  cd <- file.path(RUNS, row$formulation_id)
  tif <- file.path(cd, if (kind == "mga") "mga_guard_g05.tif" else "maa_guard_g05.tif")
  csv <- file.path(cd, if (kind == "mga") "certificates_guard.csv" else "certificates_maa_guard.csv")
  if (file.exists(tif)) { cat(sprintf("   %s/%s-guard exists -- skipped\n", row$formulation_id, kind)); return(invisible(NULL)) }
  stopifnot(file.exists(file.path(cd, "formulation_meta.json")), file.exists(file.path(cd, "anchor.tif")))
  meta <- jsonlite::read_json(file.path(cd, "formulation_meta.json"))
  wt <- form_wt(row)
  actx <- pr_override(base_for(row),
      targets = wt$t, feature_weight_multipliers = wt$w,
      results_dir = file.path(RUNS_REL, row$formulation_id),
      results_subdir = "guard_build",
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  # the frozen record STANDS: the re-solve only supplies the compiled model + x for the floors
  rel <- abs(anchor$z - meta$anchor_objective) / abs(meta$anchor_objective)
  stopifnot("re-solved anchor drifted > 1e-3 relative from formulation_meta.json -- STOP" = rel <= 1e-3)
  a_frozen <- terra::values(terra::rast(file.path(cd, "anchor.tif")))[cm$pu_index] == 1
  ham_frozen <- sum(a_frozen != anchor$x)
  cat(sprintf("   anchor re-solved %.6f vs frozen %.6f (rel %.1e) | %d cells differ from anchor.tif (near-ties)\n",
              anchor$z, meta$anchor_objective, rel, ham_frozen))
  floors <- list(ctx = actx, blocks = BLOCKS, g = FLOOR_G)
  gen <- if (kind == "mga") {
    mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, floors = floors)
  } else {
    # the seed: the manifest's e12_seed when it carries one (v4), else the E-round record's
    seed <- if ("e12_seed" %in% names(row) && !is.na(row$e12_seed)) as.integer(row$e12_seed) else ER$e12$seeds[[row$formulation_id]]
    stopifnot("no E12 seed recorded for this formulation" = !is.null(seed))
    maa_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, seed = seed, floors = floors)
  }
  layers <- lapply(seq_len(gen$k), function(i) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(r) <- v; r })
  s <- terra::rast(layers)
  names(s) <- sprintf(if (kind == "mga") "guard_%02d" else "maag_%02d", seq_len(gen$k))
  terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  write.csv(gen$certificates, csv, row.names = FALSE)
  gm <- file.path(cd, if (kind == "mga") "guard_meta.json" else "maa_guard_meta.json")
  jsonlite::write_json(list(
    formulation_id = row$formulation_id, kind = kind, floor_g = FLOOR_G, blocks = BLOCKS,
    band_g = row$band_gap_g, k = gen$k, anchor_objective_resolved = anchor$z,
    anchor_objective_frozen = meta$anchor_objective, anchor_rel_drift = rel,
    anchor_cells_differing = ham_frozen, seed = if (kind == "maa") gen$seed else NULL,
    total_runtime_s = sum(gen$certificates$runtime_s),
    wall_s = proc.time()[["elapsed"]] - t0, created_utc = format(Sys.time(), tz = "UTC")),
    gm, auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("   wrote %s (+ certificates, meta)\n", basename(tif)))
  invisible(NULL)
}


In [5]:
# ---- THE LOOP: serial over the frozen formulations, then the S0 MAA spot-check --------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  run_guard(row, "mga")
  cat(sprintf("== %s | batch elapsed %.2f h\n", row$formulation_id, (proc.time()[["elapsed"]] - t_batch) / 3600))
}
cat(sprintf("\n===================== guarded MAA spot-check: %s =====================\n", MAA_SPOTCHECK))
run_guard(MAN[MAN$formulation_id == MAA_SPOTCHECK, ], "maa")
cat("\nGUARDED SWEEP COMPLETE -- next: 12c_e19_solves, then 13, 15, 15b, 19, 20, 21\n")



===================== s0_ssp585_theta5 (1/14) =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332, F1.1.web.mix_v2.0=0.5013, F2.1.web.alt_v4.0=0.4912, T6.1.web.map_v1.0=0.3588, F1.2.web.map_v1.0=0.2451, T2.2.web.mix_v1.0=0.1476, T3.4.web.mix_v1.0=0.1094, T5.4.web.mix_v1.0=0.1046, F1.6.web.mix_v1.0=0.1009, T4.4.web.orig_v1.0=0.1, SF1.2.web.orig_v1.0=0.1, T6.2.web.alt_v2.0=0.1, T6.3.web.map_v1.0=0.1, S1.1_SF1.1.web.merged_v3=0.1, T5.1.web.mix_v1.0=0.1, F1.3.web.map_v1.0=0.1, TF1.2.web.orig_v2.0=0.1, TF1.6_TF1.7.web.merged_v3=0.1, T6.4.web.orig_v1.0=0.1, F2.4.web.mix_v1.0=0.1, T2.1.web.mix_v1.0=0.1
  override feature_weight_multipliers -> climate_type_macrorefugia=1.22468, transboundary_connectivity=0.707156, climate_corridors=1.96515, irrecoverable_carbon_m_soc=0.389745, irrecoverable_carbon_biomass=0.166555, aoh_richness_birds=1.11442, aoh_richness_mammals=1.43228
  override results_dir      -> analyses/y2y/runs_v4/s0_ssp585_theta5
  override results_s

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 1.965152)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.923851 (bound 4.923816, gap 7.26e-06) | 381,874 selected | 188 s
   anchor re-solved 4.923851 vs frozen 4.923851 (rel 7.0e-12) | 0 cells differ from anchor.tif (near-ties)
band wall appended: obj0 . x <= 5.170044  (g = 0.05 on z* = 4.923851)
  block floor core_habitat   anchor capture 0.4500 -> floor 0.4275
  block floor structural_connectivity anchor capture 0.5440 -> floor 0.5168
  block floor climate_corridors anchor capture 0.2979 -> floor 0.2830
  block floor carbon         anchor capture 0.6716 -> floor 0.6380
  block floor biodiversity   anchor capture 0.6289 -> floor 0.5975
g=0.05 iter 01/50: band 5.170045 (+5.00% of z*) OK | ham(anchor) 274,614 | 63 s
g=0.05 iter 02/50: band 5.170045 (+5.00% of z*) OK | ham(anchor) 189,672 | 63 s
g=0.05 iter 03/50: band 5.169279 (+4.98% of z*) OK | ham(anchor) 169,608 | 64 s
g=0.05 iter 04/50: band 5.155765 (+4.71% of z*) OK | ham(anc

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 2.528527)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.796701 (bound 4.796632, gap 1.43e-05) | 381,874 selected | 160 s
   anchor re-solved 4.796701 vs frozen 4.796701 (rel 6.9e-12) | 0 cells differ from anchor.tif (near-ties)
band wall appended: obj0 . x <= 5.036536  (g = 0.05 on z* = 4.796701)
  block floor core_habitat   anchor capture 0.4915 -> floor 0.4669
  block floor structural_connectivity anchor capture 0.4615 -> floor 0.4384
  block floor climate_corridors anchor capture 0.2865 -> floor 0.2721
  block floor carbon         anchor capture 0.6415 -> floor 0.6094
  block floor biodiversity   anchor capture 0.6326 -> floor 0.6009
g=0.05 iter 01/50: band 5.036490 (+5.00% of z*) OK | ham(anchor) 227,808 | 57 s
g=0.05 iter 02/50: band 5.036491 (+5.00% of z*) OK | ham(anchor) 150,770 | 58 s
g=0.05 iter 03/50: band 5.036413 (+5.00% of z*) OK | ham(anchor) 134,522 | 62 s
g=0.05 iter 04/50: band 5.036492 (+5.00% of z*) OK | ham(anc

...

compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.630268 (bound 4.630183, gap 1.84e-05) | 381,874 selected | 45 s
   anchor re-solved 4.630268 vs frozen 4.630268 (rel 9.9e-12) | 0 cells differ from anchor.tif (near-ties)
band wall appended: obj0 . x <= 4.861781  (g = 0.05 on z* = 4.630268)
  block floor core_habitat   anchor capture 0.4514 -> floor 0.4288
  block floor structural_connectivity anchor capture 0.5112 -> floor 0.4857
  block floor climate_corridors anchor capture 0.3013 -> floor 0.2862
  block floor carbon         anchor capture 0.8920 -> floor 0.8474
  block floor biodiversity   anchor capture 0.6096 -> floor 0.5792
g=0.05 iter 01/50: band 4.861781 (+5.00% of z*) OK | ham(anchor) 238,780 | 62 s
g=0.05 iter 02/50: band 4.861782 (+5.00% of z*) OK | ham(anchor) 152,976 | 60 s
g=0.05 iter 03/50: band 4.861625 (+5.00% of z*) OK | ham(anchor) 142,204 | 62 s
g=0.05 iter 04/50: band 4.861778 (+5.00% of z*) OK | ham(anch

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 10)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 11.128771 (bound 11.128633, gap 1.24e-05) | 381,874 selected | 173 s
   anchor re-solved 11.128771 vs frozen 11.128714 (rel 5.1e-06) | 3808 cells differ from anchor.tif (near-ties)
band wall appended: obj0 . x <= 11.685209  (g = 0.05 on z* = 11.128771)
  block floor core_habitat   anchor capture 0.4703 -> floor 0.4468
  block floor structural_connectivity anchor capture 0.5520 -> floor 0.5244
  block floor climate_corridors anchor capture 0.2970 -> floor 0.2822
  block floor carbon         anchor capture 0.6601 -> floor 0.6271
  block floor biodiversity   anchor capture 0.6266 -> floor 0.5953
g=0.05 iter 01/50: band 11.434974 (+2.75% of z*) OK | ham(anchor) 287,232 | 61 s
g=0.05 iter 02/50: band 11.386244 (+2.31% of z*) OK | ham(anchor) 200,836 | 61 s
g=0.05 iter 03/50: band 11.362137 (+2.10% of z*) OK | ham(anchor) 179,610 | 58 s
g=0.05 iter 04/50: band 11.374576 (+2.21% of z*)

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 1.965152)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.923851 (bound 4.923816, gap 7.26e-06) | 381,874 selected | 181 s
   anchor re-solved 4.923851 vs frozen 4.923851 (rel 7.0e-12) | 0 cells differ from anchor.tif (near-ties)
MAA band wall: obj0 . x <= 5.170044 (g = 0.05) | seed 20260910
  block floor core_habitat   anchor capture 0.4500 -> floor 0.4275
  block floor structural_connectivity anchor capture 0.5440 -> floor 0.5168
  block floor climate_corridors anchor capture 0.2979 -> floor 0.2830
  block floor carbon         anchor capture 0.6716 -> floor 0.6380
  block floor biodiversity   anchor capture 0.6289 -> floor 0.5975
maa g=0.05 iter 01/50: band 5.131496 (+4.22%) OK | ham(anchor) 171,402 | 71 s
maa g=0.05 iter 02/50: band 5.137280 (+4.33%) OK | ham(anchor) 171,458 | 70 s
maa g=0.05 iter 03/50: band 5.131654 (+4.22%) OK | ham(anchor) 171,394 | 72 s
maa g=0.05 iter 04/50: band 5.132005 (+4.23%) OK | ham(anchor) 171,266 | 

In [6]:
# ---- integrity + cost summary (results_log-ready) -------------------------------------------
tot <- 0
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  csv <- file.path(cd, "certificates_guard.csv")
  if (!file.exists(csv)) { cat(sprintf("%-22s MISSING\n", row$formulation_id)); next }
  ce <- read.csv(csv)
  gm <- file.path(cd, "guard_meta.json")
  drift <- if (file.exists(gm)) jsonlite::read_json(gm)$anchor_rel_drift else NA
  tot <- tot + sum(ce$runtime_s)
  cat(sprintf("%-22s %2d members | band_ok %s | dup %d | time-limited %d | %5.1f min | anchor drift %s\n",
              row$formulation_id, nrow(ce), if (all(ce$band_ok)) "ALL" else "VIOLATED",
              sum(ce$duplicate), sum(ce$status == "TIME_LIMIT"), sum(ce$runtime_s) / 60,
              if (is.na(drift)) "(16-era)" else sprintf("%.1e", drift)))
}
cat(sprintf("\nsweep solve time (all guarded MGA members): %.1f h\n", tot / 3600))
mcsv <- file.path(RUNS, MAA_SPOTCHECK, "certificates_maa_guard.csv")
if (file.exists(mcsv)) {
  ce <- read.csv(mcsv)
  cat(sprintf("%s guarded MAA: %d members | band_ok %s | dup %d | %.1f min | seed %d\n", MAA_SPOTCHECK,
              nrow(ce), if (all(ce$band_ok)) "ALL" else "VIOLATED", sum(ce$duplicate),
              sum(ce$runtime_s) / 60, ce$seed[1]))
}


s0_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  51.9 min | anchor drift 7.0e-12
s1_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  51.4 min | anchor drift 6.9e-12
s2_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  52.8 min | anchor drift 5.4e-12
s2c_ssp585_theta5      50 members | band_ok ALL | dup 0 | time-limited 0 |  53.3 min | anchor drift 8.2e-07
s3_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  53.6 min | anchor drift 7.3e-07
s4_ssp585_theta3       50 members | band_ok ALL | dup 0 | time-limited 0 |  52.5 min | anchor drift 9.3e-12
s5_ssp585_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  53.6 min | anchor drift 4.2e-12
s0_ssp245_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  52.5 min | anchor drift 5.3e-12
s1_ssp245_theta5       50 members | band_ok ALL | dup 0 | time-limited 0 |  52.5 min | anchor drift 1.9e-13
s2_ssp245_theta5       50 me